In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    tool = TavilySearchResults(max_results=2)
    tools = [tool]

    graph_builder = StateGraph(State)

    llm = ChatOpenAI(model_name=model_name)
    llm_with_tools = llm.bind_tools(tools)

    # チャットボットノードの作成
    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    graph_builder.add_node("chatbot", chatbot)

    tool_node = ToolNode(tools)
    graph_builder.add_node("tools", tool_node)

    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")

    graph_builder.set_entry_point("chatbot")

    memory = MemorySaver()
    graph = graph_builder.compile(checkpointer=memory)

    return graph

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values"
    )
    for event in events:
        print(event["messages"][-1].content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
while True:
    user_input = input("質問:")
    if user_input.strip() == "":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんにちは？
こんにちは！どのようにお手伝いできますか？
1たす2は？
1たす2は3です。
台湾観光について検索結果を教えて

[{"url": "https://www.hankyu-travel.com/guide/taiwan/", "content": "阪急交通社｜国内旅行・海外旅行｜ツアー予約・旅行情報サイト\n\n# 台湾観光におすすめの名所＆人気のスポットランキング台湾観光ガイド\n\n台北101\n九份\n彩虹眷村\n\n絶景にグルメ、ショッピング、温泉まで楽しめる台湾。台北のシンボルタワー、台北101でショッピングや眺望を楽しんだり、夜は寧夏夜市でB級グルメ三昧。足をのばして、九份の街歩きもおすすめです。そんな台湾の基本情報から観光情報まで詳しく紹介します。\n\n## 台湾観光マップ\n\n## 台湾のおすすめ観光スポット総合ランキング\n\n九フン\n\n### 九フン\n\n『千と千尋の神隠し』の舞台として脚光を浴びた山間部にある小さな町。赤提灯が並ぶ風景を求め、大勢の観光客が訪れます。\n\n士林夜市\n\n### 士林夜市\n\n台北市最大規模の観光夜市です。台湾のローカルフードを提供する飲食屋台などさまざまな屋台が軒を連ねます。\n\n日月潭\n\n### 日月潭\n\n周囲を山々に囲まれた台湾最大の淡水湖。台湾3大観光地の一つとして人気で、四季折々の美しい景観が楽しめます。\n\n台北101\n\n### 台北101\n\n地上101階建て、高さ508mの超高層ビル。毎年元旦0時にはカウントダウン花火を実施しています。\n\n野柳\n\n### 野柳\n\n全長約1,700mの長さの岬。海食風化と地殻変動により、大自然の石彫芸術のような独特の景観を生み出しました。\n\n龍虎塔\n\n### 龍虎塔\n\n1976年に建てられた7階建の塔です。龍の口から入り、虎の口から出るルールがあり、福を得て、凶を避けられます。\n\n太魯閣溪谷\n\n### 太魯閣溪谷\n\n太魯閣国立公園の中にある、台湾八景の一つとして知られる渓谷。高山、断崖、滝など豊かな自然を感じられます。\n\n台湾おすすめツアー\n\n## 台湾のおすすめ観光スポット\n\n#### テーマで絞り込む\n\n#### エリアで絞り込む\n\n九份 [...] 台湾のグ